In [1]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from datetime import timedelta
import streamlit as st


In [2]:
DATA_ATUAL_SIMULADA_ESTOQUE = pd.to_datetime("2025-06-16").date()  # Data simulada para o estoque

df = pd.read_csv('base/vendas.csv')
mov = pd.read_csv('base/movestoque.csv')
embalagem = pd.read_csv('base/embalagem.csv')
formapgto = pd.read_csv('base/formapgto.csv')
produto = pd.read_csv('base/produto.csv')
classificacao = pd.read_csv('base/classificacao2.csv')


C:\Users\leopa\AppData\Local\Temp\ipykernel_23188\3101149667.py:3: DtypeWarning: Columns (14) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('base/vendas.csv')


In [3]:
estoque = mov[['descricao_movimentacao', 'embalagemid',
       'unidadenegocioid', 'datahora','quantidade',
       'estoqueanterior', 'estoque', 'precoreferencial', 'precovenda',
       'customedio']].copy()
estoque = estoque.merge(embalagem[['id', 'produtoid', 'apresentacao', 'codigobarras', 
                         'markup', 'descricao']],
                         left_on='embalagemid', right_on='id', how='left')
estoque = estoque.merge(classificacao,
                        on='produtoid',
                        how='left'
                        )
estoque.rename(columns={'nome': 'classificacao_nome'}, inplace=True)

mapping = {
    93733: "DROGARIA JB - F01 - MATRIZ",
    93729: "DROGARIA JB - F02 - CAMERINO",
    93730: "DROGARIA JB - F03 - AILKA"
}

estoque["filial_nome"] = estoque["unidadenegocioid"].map(mapping)

mapping2 = {
    93733: 1,
    93729: 2,
    93730: 3
} 
estoque["filial_codigo"] = estoque["unidadenegocioid"].map(mapping2)
estoque = estoque[estoque['unidadenegocioid'].isin(mapping.keys())]

def extrair_class_painome(caminho):
    if caminho is None or pd.isna(caminho):
        return "Sem classificação"
    if 'QUERODELIVERY' in caminho:
        return 'QUERODELIVERY'
    partes = caminho.split(' > ')
    return partes[1] if len(partes) > 1 else None

estoque['class_painome'] = estoque['caminho'].apply(extrair_class_painome)

In [4]:
class_estoque = ['VAREJO', 'PRESCRIÇÃO', 'INDICAÇÃO', 'SBB']
estoque = estoque[estoque['class_painome'].isin(class_estoque)]

In [5]:
# Converter datahora para datetime se necessário
estoque['datahora'] = pd.to_datetime(estoque['datahora'])

# Filtrar vendas dos últimos 90 dias
data_limite_90 = pd.Timestamp(DATA_ATUAL_SIMULADA_ESTOQUE) - pd.Timedelta(days=90)
vendas_90d = estoque[
    (estoque['descricao_movimentacao'] == 'Venda') &
    (estoque['datahora'] >= data_limite_90)
].copy()

# Converter quantidade para positiva (vendas são negativas)
vendas_90d['quantidade'] = vendas_90d['quantidade'].abs()

In [25]:
# Agrupar por filial, class_painome e embalagemid, depois somar quantidades
vendas_produto = (vendas_90d.groupby(['filial_nome', 'classificacao_nome', 'embalagemid','descricao'])['quantidade']
                  .sum()
                  .reset_index())

# Função para classificar curva ABC
def classificar_curva(perc):
    if perc <= 50:
        return 'A'
    elif perc <= 80:
        return 'B'
    else:
        return 'C'

# Calcular curva ABC para cada combinação filial + class_painome
resultado_curvas = []

for filial in vendas_produto['filial_nome'].unique():
    for categoria in vendas_produto['classificacao_nome'].unique():
        
        # Filtrar dados para esta filial e categoria
        dados_filtrados = vendas_produto[
            (vendas_produto['filial_nome'] == filial) & 
            (vendas_produto['classificacao_nome'] == categoria)
        ].copy()
        
        if len(dados_filtrados) > 0:
            # Ordenar por quantidade (decrescente)
            dados_filtrados = dados_filtrados.sort_values('quantidade', ascending=False)
            
            # Calcular percentual acumulado
            dados_filtrados['quantidade_acum'] = dados_filtrados['quantidade'].cumsum()
            total_vendas_categoria = dados_filtrados['quantidade'].sum()
            dados_filtrados['perc_acum'] = (dados_filtrados['quantidade_acum'] / total_vendas_categoria) * 100
            
            # Classificar em curvas ABC
            dados_filtrados['curvaABC'] = dados_filtrados['perc_acum'].apply(classificar_curva)
            
            # Adicionar ao resultado
            resultado_curvas.append(dados_filtrados[['filial_nome', 'classificacao_nome', 'embalagemid', 'descricao','quantidade','quantidade_acum','perc_acum','curvaABC']])

# Combinar todos os resultados
if resultado_curvas:
    vendas_com_curva = pd.concat(resultado_curvas, ignore_index=True)
else:
    vendas_com_curva = pd.DataFrame(columns=['filial_nome', 'classificacao_nome', 'embalagemid', 'curvaABC'])

In [29]:
# Produtos que tiveram vendas nos últimos 90 dias (manter filial e categoria)
produtos_com_vendas = vendas_com_curva[['filial_nome', 'classificacao_nome', 'embalagemid', 'curvaABC']]

# Todos os produtos únicos no estoque por filial e categoria
todos_produtos_filial_categoria = estoque[['filial_nome', 'classificacao_nome', 'embalagemid']].drop_duplicates()

# Fazer merge para identificar produtos sem vendas
produtos_com_vendas_merge = produtos_com_vendas[['filial_nome', 'classificacao_nome', 'embalagemid']].copy()
produtos_sem_vendas = todos_produtos_filial_categoria.merge(
    produtos_com_vendas_merge, 
    on=['filial_nome', 'classificacao_nome', 'embalagemid'], 
    how='left', 
    indicator=True
)

# Filtrar apenas os que não tiveram vendas
produtos_sem_vendas = produtos_sem_vendas[produtos_sem_vendas['_merge'] == 'left_only']
produtos_sem_vendas = produtos_sem_vendas[['filial_nome', 'classificacao_nome', 'embalagemid']].copy()
produtos_sem_vendas['curvaABC'] = 'D'

# Combinar produtos com vendas e sem vendas
curva_abc_final = pd.concat([produtos_com_vendas, produtos_sem_vendas], ignore_index=True)

In [30]:
# Fazer merge com o estoque usando filial_nome, class_painome e embalagemid
estoque_com_curva = estoque.merge(
    curva_abc_final, 
    on=['filial_nome', 'classificacao_nome', 'embalagemid'], 
    how='left'
)

# Produtos não encontrados ficam como 'D'
estoque_com_curva['curvaABC'] = estoque_com_curva['curvaABC'].fillna('D')

In [31]:
estoque_com_curva[['embalagemid', 'unidadenegocioid', 
       'quantidade', 'estoqueanterior', 'estoque', 'produtoid', 'apresentacao',
       'codigobarras','descricao', 'classificacaoid',
       'classificacao_nome', 'class_painome', 'curvaABC']].to_csv('base/estoque_curva_abc.csv', sep=';',encoding='latin1',index=False)

In [32]:
data_limite_90 = pd.Timestamp(DATA_ATUAL_SIMULADA_ESTOQUE) - pd.Timedelta(days=90)
estoque_90d = estoque_com_curva[estoque_com_curva['datahora'] >= data_limite_90].copy()

In [33]:
# Obter estoque atual (incluir class_painome)
estoque_atual = (estoque_90d.sort_values('datahora')
                .groupby(['filial_nome', 'embalagemid'])
                .tail(1)[['filial_nome', 'embalagemid', 'class_painome', 'curvaABC', 'estoque']])


In [34]:
# 1) Índice de ruptura por filial e curva (o que já existe)
resultado_filial_curva = []

for filial in estoque_atual['filial_nome'].unique():
    for curva in ['A', 'B', 'C', 'D']:
        produtos_curva = estoque_atual[
            (estoque_atual['filial_nome'] == filial) & 
            (estoque_atual['curvaABC'] == curva)
        ]
        
        if len(produtos_curva) > 0:
            total_produtos = len(produtos_curva)
            produtos_zerados = len(produtos_curva[produtos_curva['estoque'] == 0])
            indice_ruptura = (produtos_zerados / total_produtos) * 100
            
            resultado_filial_curva.append({
                'filial_nome': filial,
                'curvaABC': curva,
                'total_produtos': total_produtos,
                'produtos_zerados': produtos_zerados,
                'indice_ruptura': indice_ruptura
            })

In [36]:
# 2) Índice de ruptura apenas por filial
resultado_filial = []

for filial in estoque_atual['filial_nome'].unique():
    produtos_filial = estoque_atual[estoque_atual['filial_nome'] == filial]
    
    if len(produtos_filial) > 0:
        total_produtos = len(produtos_filial)
        produtos_zerados = len(produtos_filial[produtos_filial['estoque'] == 0])
        indice_ruptura = (produtos_zerados / total_produtos) * 100
        
        resultado_filial.append({
            'filial_nome': filial,
            'total_produtos': total_produtos,
            'produtos_zerados': produtos_zerados,
            'indice_ruptura': indice_ruptura
        })


In [37]:
# 3) Índice de ruptura por filial e class_painome
resultado_filial_categoria = []

for filial in estoque_atual['filial_nome'].unique():
    for categoria in estoque_atual['class_painome'].unique():
        produtos_categoria = estoque_atual[
            (estoque_atual['filial_nome'] == filial) & 
            (estoque_atual['class_painome'] == categoria)
        ]
        
        if len(produtos_categoria) > 0:
            total_produtos = len(produtos_categoria)
            produtos_zerados = len(produtos_categoria[produtos_categoria['estoque'] == 0])
            indice_ruptura = (produtos_zerados / total_produtos) * 100
            
            resultado_filial_categoria.append({
                'filial_nome': filial,
                'class_painome': categoria,
                'total_produtos': total_produtos,
                'produtos_zerados': produtos_zerados,
                'indice_ruptura': indice_ruptura
            })

In [38]:
# 4) Índice de ruptura por filial, class_painome e curva
resultado_filial_categoria_curva = []

for filial in estoque_atual['filial_nome'].unique():
    for categoria in estoque_atual['class_painome'].unique():
        for curva in ['A', 'B', 'C', 'D']:
            produtos_completo = estoque_atual[
                (estoque_atual['filial_nome'] == filial) & 
                (estoque_atual['class_painome'] == categoria) &
                (estoque_atual['curvaABC'] == curva)
            ]
            
            if len(produtos_completo) > 0:
                total_produtos = len(produtos_completo)
                produtos_zerados = len(produtos_completo[produtos_completo['estoque'] == 0])
                indice_ruptura = (produtos_zerados / total_produtos) * 100
                
                resultado_filial_categoria_curva.append({
                    'filial_nome': filial,
                    'class_painome': categoria,
                    'curvaABC': curva,
                    'total_produtos': total_produtos,
                    'produtos_zerados': produtos_zerados,
                    'indice_ruptura': indice_ruptura
                })

# Converter para DataFrames
df_ruptura_filial_curva = pd.DataFrame(resultado_filial_curva)
df_ruptura_filial = pd.DataFrame(resultado_filial)
df_ruptura_filial_categoria = pd.DataFrame(resultado_filial_categoria)
df_ruptura_filial_categoria_curva = pd.DataFrame(resultado_filial_categoria_curva)


In [35]:
df_ruptura_filial.to_csv('base/ruptura_filial.csv', sep=';',decimal=',',encoding='latin1',index=False)
df_ruptura_filial_curva.to_csv('base/ruptura_filial_curva.csv', sep=';',decimal=',',encoding='latin1',index=False)
df_ruptura_filial_categoria.to_csv('base/ruptura_filial_categoria.csv',sep=';',decimal=',',encoding='latin1', index=False)
df_ruptura_filial_categoria_curva.to_csv('base/ruptura_filial_categoria_curva.csv', sep=';',decimal=',',encoding='latin1',index=False)